In [ ]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


In [ ]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

In [ ]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_7'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");


In [ ]:
%%jshell

runPy("""
# Import required libraries
import numpy as np

from sklearn.decomposition import PCA # Class that performs PCA
from sklearn.datasets import load_breast_cancer # To load the dataset on the fly
from sklearn.preprocessing import StandardScaler # To standardize the data

import matplotlib.pyplot as plt # Visualization
""");


In [ ]:
%%jshell

runPy("""
# Dimensionality Reduction using Principal Component Analysis (PCA)
#
# In this lab, we perform Principal Component Analysis on the breast cancer
# dataset.
#
# PCA is a technique used to reduce the dimensionality of a dataset while
# retaining as much of the original variance as possible.
#
# This can be useful for visualizing high-dimensional data or for improving the
# performance of machine learning algorithms by removing noise and
# multicollinearity.
#
# We have primarily used it here to visualize the
""");


In [ ]:
%%jshell

runPy("""
# Load the Breast Cancer dataset
dataset = load_breast_cancer()
dataset.data
dataset.target
""");


## Dataset Description and PCA's Role

This dataset contains information about breast cell nuclei, with each sample described by 30 measurements (features). These features include things like the size, shape, and texture of the cell nuclei. The goal is to determine if a mass is malignant (cancerous) or benign (not cancerous) based on these features.

Having 30 features makes it hard to visualize the data directly. Principal Component Analysis (PCA) helps by reducing these 30 features to a smaller number, in this case, two main components. PCA finds the directions in the data where there is the most variation.

By using PCA to reduce the data to 2 components, we can:

1.  **Simplify the data:** Make it easier to see patterns by going from 30 dimensions to just 2.
2.  **See the separation:** The plot shows how well the first two components separate the malignant and benign cases, indicating that PCA captures the key differences.
3.  **Reduce noise:** Focus on the most important variations in the data, ignoring less important ones.

PCA makes this high-dimensional breast cancer data easier to understand and visualize, especially for seeing the difference between cancerous and non-cancerous cells.

In [ ]:
%%jshell

runPy("""
# Get the data from the dataset
from sklearn.datasets import load_breast_cancer # Ensure import for dataset loading
dataset = load_breast_cancer() # Added to ensure dataset is defined
data = dataset.data

# Display the (number of rows, dimensionality of each row)
data.shape
""");


In [ ]:
%%jshell

runPy("""
# Standardize the data
#
# Standardizing the data is important for PCA because it is sensitive to the
# scale of the features.
#
# Print the scaled data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)
print(scaled_data)
scaled_data.shape
""");


In [ ]:
%%jshell

runPy("""
# Create a PCA instance with 2 components
#
# We choose 2 components to reduce the data to a 2-dimensional space,
#  which is useful for visualization.
n_components = 2
pca = PCA(n_components=n_components)
""");


In [ ]:
%%jshell

runPy("""
import pandas as pd

# Get the feature names from the dataset
feature_names = dataset.feature_names

# Fit PCA to the scaled data if it hasn't been fitted already
# This step calculates the principal components based on the scaled data.
pca.fit(scaled_data)

# Create a DataFrame to better visualize the principal components' loadings
# Each row is a principal component, and each column is an original feature
components_df = pd.DataFrame(pca.components_, columns=feature_names, index=[f'Principal Component {i+1}' for i in range(n_components)])

print("Principal Components Loadings:")
display(components_df)
""");


The table above shows the loadings (coefficients) for each original feature on Principal Component 1 and Principal Component 2. A higher absolute value for a coefficient indicates that the corresponding original feature has a stronger influence on that principal component.

For example, if 'mean radius' has a high positive loading on Principal Component 1, it means that PC1 increases as 'mean radius' increases. If it has a high negative loading, PC1 decreases as 'mean radius' increases.

By examining these values, you can infer which original features are most strongly associated with each principal component and, consequently, which features contribute most to the variance captured by that component.

In [ ]:
%%jshell

runPy("""
# Fit PCA to the scaled data
#
# This step calculates the principal components based on the scaled data.
pca.fit(scaled_data)
""");


In [ ]:
%%jshell

runPy("""
# Transform the data to its principal components
#
# This step applies the calculated principal components to the scaled data to reduce its dimensionality.
transformed_data = pca.transform(scaled_data)
transformed_data.shape
""");


In [ ]:
%%jshell

runPy("""
# Explained variance ratio
# The explained variance ratio indicates the proportion of the dataset's variance that lies along each principal component.
explained_variance_ratio = pca.explained_variance_ratio_
print("Explained Variance Ratio:", explained_variance_ratio)

# Principal components
# The principal components are the directions in the original feature space along which the variance is maximized.
principal_components = pca.components_
print("Principal Components:", principal_components)
""");


In [ ]:
%%jshell

runPy("""
# Load the target variable for coloring the plot
target = dataset.target

# Create a scatter plot of the transformed data
plt.figure(figsize=(8, 6))
plt.scatter(transformed_data[:, 0], transformed_data[:, 1], c=target, cmap='viridis')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of Breast Cancer Dataset (2 Components)')
plt.colorbar(label='Target (0: Malignant, 1: Benign)')
plt.grid(True)
plt.savefig('/content/regression_plot.png')
""");


In [ ]:
%%jshell

runPy("""
# Load the target variable for coloring the plot
target = dataset.target

# Create a scatter plot of the transformed data
plt.figure(figsize=(8, 6))

# Separate data by target class for legend
transformed_data_malignant = transformed_data[target == 0]
transformed_data_benign = transformed_data[target == 1]

# Plot each class separately
plt.scatter(transformed_data_malignant[:, 0], transformed_data_malignant[:, 1], c='purple', label='Malignant (0)', alpha=0.7)
plt.scatter(transformed_data_benign[:, 0], transformed_data_benign[:, 1], c='yellow', label='Benign (1)', alpha=0.7)

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of Breast Cancer Dataset (2 Components)')
plt.legend() # Add the legend
plt.grid(True)
plt.savefig('/content/regression_plot.png')
""");


In [ ]:
%%jshell

runPy("""
# Final interpretation:

# The scatter plot shows a clear separation between the two target classes
# (Malignant and Benign) in the 2-dimensional space created by the first two
# principal components.
#
# This suggests that these two components capture a
# significant amount of the variance that differentiates the two classes,
# making it much easier to visualize the data and potentially build a
# classification model on this reduced-dimensional data.

# The explained variance ratio shows that the first principal component captures
# about 44.3% of the variance, and the second captures about 19.0%. Together,
# they capture approximately 63.3% of the total variance in the original
# 30-dimensional dataset.

# While this doesn't capture all the variance, the clear separation in the plot
# indicates that the most important information for distinguishing between the
# two classes is largely preserved in these two components.

# Conclusion: PCA has transformed the 30-dimensional data into a 2-dimensional
# representation where the classes are visually distinguishable, which is a
# meaningful outcome for understanding the data structure and potentially
# for downstream tasks like classification.
""");


In [ ]:
%%jshell

runPy("""
import pandas as pd

# Create a DataFrame from the transformed data
# transformed_data and target are available from previous steps.
df = pd.DataFrame(data=transformed_data, columns=['Principal Component 1', 'Principal Component 2'])

# Add the target variable to the DataFrame
df['Target'] = target

df.to_excel('final_dataset.xlsx',index=False)
""");


In [ ]:
%%jshell

runPy("""
import pandas as pd

# Create a DataFrame from the transformed data
df = pd.DataFrame(data=transformed_data, columns=['Principal Component 1', 'Principal Component 2'])

# Add the target variable to the DataFrame
df['Target'] = target

# Display the first few rows of the new DataFrame
display(df.head())
""");


In [ ]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");
